# Optimization for Training Deep Models — a hands-on MNIST notebook

*Companion to Chapter 8 of Goodfellow, Bengio & Courville — "Optimization for Training Deep Models."*

Training a network is optimization, but an unusual kind: we descend a noisy, high-dimensional loss landscape riddled with saddle points, plateaus, cliffs and gradients that vanish through depth — and we don't even want the true minimum, we want parameters that *generalize*. This notebook turns each "station" of the lesson into a runnable MNIST experiment.

| # | Station | What we run on MNIST |
|---|---------|----------------------|
| 1 | Learning vs optimization | Surrogate losses; train-loss ↓ while val-loss ↑ (early stopping) |
| 2 | Optimization challenges | Ill-conditioning zig-zag, a saddle that stalls, a cliff + clipping |
| 3 | Depth bites back | Per-layer gradient norms: sigmoid **vanishes**, big init **explodes** |
| 4 | Stochastic gradient descent | Learning-rate sweep, batch-noise ∝ 1/√b, LR decay |
| 5 | Momentum & Nesterov | 2-D valley race + momentum on MNIST |
| 6 | Initialization | Zero-init kills learning; variance preserved at the He scale |
| 7 | Adaptive rates · Adam | SGD vs AdaGrad vs RMSProp vs Adam, on a bowl and on MNIST |
| 8 | Greedy pretraining | Stacked-autoencoder init beats random in the low-label regime |
| 9 | Adversarial training | FGSM breaks a clean model; adversarial training hardens it |

> **The one idea to carry off the map:** training is *noisy descent through a hostile landscape*. The gradient is estimated (SGD), consistent directions are worth accelerating (momentum), each parameter deserves its own step (Adam), and where you start (initialization) decides whether the signal even survives the depth.

**Runtime:** works on CPU; a GPU (Runtime → Change runtime type → GPU) is noticeably faster for stations 3–9. Run cells top to bottom.


## Setup — data, models, and one training loop

We load MNIST once and reuse a single configurable `DeepMLP` (any depth/width, choice of activation and initialization) plus one `train()` loop across every station. Read the comments — they explain the *why*.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, TensorDataset

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.1307,), (0.3081,))])
train_full = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_full  = datasets.MNIST('./data', train=False, download=True, transform=transform)
print('train', len(train_full), '| test', len(test_full))

In [ ]:
# A moderate training subset keeps every experiment fast but still meaningful.
idx6 = np.random.RandomState(0).choice(len(train_full), 6000, replace=False)
train_loader = DataLoader(Subset(train_full, idx6), batch_size=128, shuffle=True)
val_loader   = DataLoader(test_full, batch_size=512, shuffle=False)

def make_loader(n, bs=64, seed=1, train=True):
    ds = train_full if train else test_full
    idx = np.random.RandomState(seed).choice(len(ds), n, replace=False)
    return DataLoader(Subset(ds, idx), batch_size=bs, shuffle=train)

print('main train subset:', len(idx6), 'images')

In [ ]:
class DeepMLP(nn.Module):
    """Configurable MLP used across the whole notebook.
       sizes=[784, ..., 10]; act in {relu, tanh, sigmoid}; forward can return activations."""
    def __init__(self, sizes, act='relu', init='he'):
        super().__init__()
        self.lins = nn.ModuleList([nn.Linear(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)])
        self.act_name = act
        self.reset(init)

    def act(self, x):
        return {'relu': torch.relu, 'tanh': torch.tanh, 'sigmoid': torch.sigmoid}[self.act_name](x)

    def reset(self, init='he', scale=1.0):
        for lin in self.lins:
            if   init == 'zero':   nn.init.zeros_(lin.weight)
            elif init == 'he':     nn.init.kaiming_normal_(lin.weight, nonlinearity='relu'); lin.weight.data.mul_(scale)
            elif init == 'xavier': nn.init.xavier_normal_(lin.weight); lin.weight.data.mul_(scale)
            elif init == 'normal': nn.init.normal_(lin.weight, 0.0, scale)
            nn.init.zeros_(lin.bias)
        return self

    def forward(self, x, return_acts=False):
        x = x.view(x.size(0), -1)
        acts = []
        for i, lin in enumerate(self.lins):
            x = lin(x)
            if i < len(self.lins) - 1:
                x = self.act(x)
            acts.append(x)
        return (x, acts) if return_acts else x

In [ ]:
def build_opt(model, name='sgd', lr=0.1, momentum=0.0, nesterov=False, weight_decay=0.0):
    p = model.parameters()
    if name == 'sgd':     return torch.optim.SGD(p, lr=lr, momentum=momentum, nesterov=nesterov, weight_decay=weight_decay)
    if name == 'adagrad': return torch.optim.Adagrad(p, lr=lr)
    if name == 'rmsprop': return torch.optim.RMSprop(p, lr=lr)
    if name == 'adam':    return torch.optim.Adam(p, lr=lr)
    if name == 'adamw':   return torch.optim.AdamW(p, lr=lr, weight_decay=weight_decay)
    raise ValueError(name)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); tot = correct = 0; loss_sum = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)                                   # DeepMLP flattens internally
        loss_sum += F.cross_entropy(out, yb, reduction='sum').item()
        correct  += (out.argmax(1) == yb).sum().item(); tot += yb.size(0)
    return loss_sum / tot, correct / tot

def train(model, loader, epochs=8, opt_name='sgd', lr=0.1, momentum=0.0, nesterov=False,
          weight_decay=0.0, val_loader=val_loader, clip=None, record_steps=False, log_every=0):
    model.to(device)
    opt = build_opt(model, opt_name, lr, momentum, nesterov, weight_decay)
    hist = {k: [] for k in ['train_loss','train_acc','val_loss','val_acc','steps']}
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            if clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            if record_steps:
                hist['steps'].append(loss.item())
        tl, ta = evaluate(model, loader); hist['train_loss'].append(tl); hist['train_acc'].append(ta)
        if val_loader is not None:
            vl, va = evaluate(model, val_loader); hist['val_loss'].append(vl); hist['val_acc'].append(va)
        if log_every and (ep+1) % log_every == 0:
            print(f'  epoch {ep+1:3d} | train {ta:.3f} | val {hist["val_acc"][-1]:.3f}')
    return hist

## Station 1 — How learning differs from optimization

Optimization minimizes a function; learning wants a function it *can't measure*. We would love to minimize the **true risk** `J*(θ) = E_{(x,y)∼p_data}[L(f(x;θ),y)]`, but we only have samples, so we minimize the **empirical risk** over the training set and hope they agree. Three more gaps follow:

- **Surrogate loss.** The metric we care about — 0–1 classification error — has *zero gradient almost everywhere*. We optimize a smooth surrogate (cross-entropy / log-loss) that upper-bounds it, then *report* accuracy.
- **Early stopping.** Optimization halts when the gradient is small; learning halts when a *validation* metric stops improving — usually long before training loss bottoms out.
- **Minibatches.** We estimate the gradient from a small random batch — many cheap, noisy steps instead of one exact one.

First, why the surrogate: plot the common classification losses against the margin `m = y·f(x)`.


In [ ]:
m = np.linspace(-3, 3, 500)
zero_one = (m < 0).astype(float)                 # 1 if wrong, 0 if right — flat gradient
logistic = np.log(1 + np.exp(-m))                # smooth, never quite 0
hinge    = np.maximum(0, 1 - m)                  # SVM: 0 once margin > 1
squared  = (1 - m)**2                            # penalizes even over-confident-correct

plt.figure(figsize=(8,5))
plt.step(m, zero_one, where='post', color='gray',    lw=2, label='0–1 loss (no gradient)')
plt.plot(m, logistic,                 color='purple', lw=2, label='logistic / log-loss')
plt.plot(m, hinge,                    color='teal',   lw=2, label='hinge / SVM')
plt.plot(m, squared,                  color='crimson',lw=2, label='squared')
plt.axvline(0, color='k', lw=.5); plt.ylim(0, 4)
plt.xlabel('margin  m = y·f(x)   (right of 0 = correct)'); plt.ylabel('loss')
plt.title('Station 1 — every surrogate sits above the 0–1 step, but has a usable gradient')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print('The 0–1 loss is flat -> gradient 0 almost everywhere. We optimize a smooth surrogate instead.')

### Train-loss falls while validation-loss turns up

This mismatch is why the training and validation curves **diverge**: training loss keeps sliding as the optimizer overfits the finite sample, while validation loss bottoms out and turns back up. You watch the validation curve, not the training loss — that's **early stopping**. We reproduce it on a small subset.


In [ ]:
torch.manual_seed(0)
model_es = DeepMLP([784, 256, 256, 10], act='relu', init='he')
small = make_loader(1000, bs=64)
hist_es = train(model_es, small, epochs=80, opt_name='sgd', lr=0.05, val_loader=val_loader)

best = int(np.argmin(hist_es['val_loss']))
fig, ax = plt.subplots(1, 2, figsize=(13,4.6))
ax[0].plot(hist_es['train_loss'], label='train'); ax[0].plot(hist_es['val_loss'], label='val')
ax[0].axvline(best, color='crimson', ls='--', label=f'early stop (epoch {best})')
ax[0].set_title('Loss: train keeps falling, val turns up'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(hist_es['train_acc'], label='train'); ax[1].plot(hist_es['val_acc'], label='val')
ax[1].axvline(best, color='crimson', ls='--'); ax[1].set_title('Accuracy (the metric we report)')
ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()
print(f'Best validation loss at epoch {best}; val acc there = {hist_es["val_acc"][best]:.3f}. '
      f'Train loss keeps dropping past it — that later progress is overfitting.')

## Station 2 — Challenges in neural-network optimization

Loss surfaces fight back. The classic worry was local minima, but in high dimensions the real obstacles are **ill-conditioning, saddle points, plateaus and cliffs**. We illustrate each on tiny 2-D surfaces where we can *see* gradient descent struggle — the diagnostics transfer directly to real training.


### (a) Ill-conditioning — the zig-zag

When the Hessian has wildly different curvature across directions (large **condition number** `κ = λ_max/λ_min`), gradient descent bounces across the steep direction while crawling along the flat one. The step size is throttled by the steep direction, so the flat direction barely moves.


In [ ]:
def gd_quadratic(kappa, lr, steps=40, start=(-4.0, 4.0)):
    H = np.array([kappa, 1.0])                 # curvature per axis (diagonal Hessian)
    w = np.array(start, float); path = [w.copy()]
    for _ in range(steps):
        w = w - lr * (H * w)                   # grad of 1/2 w^T H w  is  H w
        path.append(w.copy())
    return np.array(path)

kappa = 20.0
path = gd_quadratic(kappa, lr=2.0/(kappa+1))   # largest stable-ish rate
g = np.linspace(-5, 5, 200); W1, W2 = np.meshgrid(g, g)
J = 0.5*(kappa*W1**2 + W2**2)
plt.figure(figsize=(6.5,6))
plt.contour(W1, W2, J, levels=np.logspace(-1, 3, 20), cmap='viridis', alpha=0.6)
plt.plot(path[:,0], path[:,1], 'o-', color='crimson', ms=3, lw=1, label='GD iterates')
plt.scatter(0,0,color='k',zorder=5,label='minimum')
plt.title(f'Station 2a — ill-conditioning (κ={kappa:.0f}): GD zig-zags')
plt.xlabel('w₁ (steep)'); plt.ylabel('w₂ (flat)'); plt.legend(); plt.show()
print(f'condition number κ = {kappa:.0f}. Momentum & adaptive methods (stations 5, 7) target exactly this.')

### (b) Saddle points — where first-order methods stall

In high dimensions, almost all critical points are **saddles**, not minima: the surface goes down in some directions and up in others, and right at the saddle the gradient nearly vanishes — so gradient descent *crawls*. Yet the loss there is far above the minimum, so escaping matters. Watch `|gradient|` collapse as the iterate approaches a saddle.


In [ ]:
# f(w) = w1^2 - w2^2  ->  saddle at origin; grad = (2 w1, -2 w2)
w = np.array([2.0, 1e-3]); lr = 0.05; gnorm = []
for _ in range(80):
    grad = np.array([2*w[0], -2*w[1]])
    gnorm.append(np.linalg.norm(grad))
    w = w - lr*grad
plt.figure(figsize=(7.5,4.6))
plt.plot(gnorm, color='crimson', lw=2)
plt.xlabel('step'); plt.ylabel('|gradient|')
plt.title('Station 2b — approaching a saddle: the gradient (and progress) collapses')
plt.grid(alpha=0.3); plt.show()
print('Loss flat but gradient tiny is the signature of a saddle/plateau — momentum helps punch through.')

### (c) Cliffs & gradient clipping

Sharp cliffs — common in RNNs — produce a huge gradient that flings the parameters far away in a single step. **Gradient clipping** rescales any gradient whose norm exceeds a threshold, so one spike can't wreck the run.


In [ ]:
def clip_norm(g, thresh):
    n = np.linalg.norm(g)
    return g * (thresh / n) if n > thresh else g

g_spike = np.array([300.0, -40.0])              # gradient at the foot of a cliff
lr = 0.05
step_raw     = lr * g_spike
step_clipped = lr * clip_norm(g_spike, thresh=5.0)
print(f'raw gradient norm      : {np.linalg.norm(g_spike):8.1f}')
print(f'step WITHOUT clipping  : length {np.linalg.norm(step_raw):8.2f}  <- flies off the cliff')
print(f'step WITH clip@5.0     : length {np.linalg.norm(step_clipped):8.2f}  <- safe, same direction')
print('\ntorch.nn.utils.clip_grad_norm_(model.parameters(), max_norm) does exactly this in training.')

## Station 3 — The difficulty of training deep networks

Back-propagation multiplies. The gradient reaching layer 1 is the output gradient times a product of every layer's weight matrix and activation slope:

$$\frac{\partial L}{\partial h_1} = W_L^\top\,\mathrm{diag}(\sigma')\cdots W_2^\top\,\mathrm{diag}(\sigma')\cdot\frac{\partial L}{\partial h_L}\ \Rightarrow\ \text{magnitude}\approx (w\cdot\sigma')^L.$$

A per-layer factor below 1 makes the signal **vanish**; above 1 makes it **explode** — either way the early layers get a useless gradient. Sigmoid's slope maxes at 0.25, so it fades fast; ReLU keeps slope 1 on the active side. We measure the **actual per-layer gradient norms** in a 12-layer MLP for several activation/init choices.


In [ ]:
def per_layer_grad_norms(sizes, act, init, scale=1.0):
    torch.manual_seed(0)
    model = DeepMLP(sizes, act=act, init=init).to(device)
    model.reset(init, scale=scale)               # apply the requested init scale
    xb, yb = next(iter(train_loader)); xb, yb = xb.to(device), yb.to(device)
    F.cross_entropy(model(xb), yb).backward()
    return [lin.weight.grad.norm().item() for lin in model.lins]

sizes = [784] + [128]*11 + [10]                  # 12 weight layers
configs = [
    ('sigmoid, xavier',      'sigmoid', 'xavier', 1.0, 'gray'),
    ('relu, He (good)',      'relu',    'he',     1.0, 'green'),
    ('relu, He × 2 (big)',   'relu',    'he',     2.0, 'crimson'),
]
plt.figure(figsize=(8.5,5))
for label, act, init, scale, c in configs:
    gn = per_layer_grad_norms(sizes, act, init, scale)
    plt.semilogy(range(1, len(gn)+1), gn, 'o-', color=c, label=label)
plt.xlabel('layer (1 = input side, deepest from the loss)'); plt.ylabel('‖grad‖ (log scale)')
plt.title('Station 3 — gradient magnitude through depth')
plt.legend(); plt.grid(alpha=0.3, which='both'); plt.show()
print('sigmoid: gradient decays toward the input layers (vanishing).')
print('ReLU+He: roughly level. ReLU+big init: grows toward the input layers (exploding).')

## Station 4 — Stochastic gradient descent

SGD estimates the gradient from a small random **minibatch**, steps, and repeats:

$$\hat g=\tfrac1b\sum_{i\in\text{batch}}\nabla_\theta L,\qquad \theta\leftarrow\theta-\varepsilon_k\hat g.$$

Two knobs decide everything: the **learning rate** `ε` (too small → glacial, too big → diverge) and the **batch size** `b` (noise ∝ 1/√b). The noise is useful — it jostles the iterate out of shallow traps — but it forces the rate to **decay** so the run can settle (`Σεₖ=∞, Σεₖ²<∞`).


In [ ]:
# --- learning-rate sweep: watch too-small crawl and too-big diverge ---
def smooth(x, k=15):
    x = np.array(x); 
    return np.convolve(x, np.ones(k)/k, mode='valid') if len(x) >= k else x

plt.figure(figsize=(8.5,5))
for lr, c in [(1e-3,'navy'), (5e-2,'green'), (5e-1,'orange'), (2.0,'crimson')]:
    torch.manual_seed(0)
    m = DeepMLP([784, 256, 10], init='he')
    h = train(m, train_loader, epochs=6, opt_name='sgd', lr=lr, record_steps=True, val_loader=None)
    s = np.array(h['steps'])
    s = np.where(np.isfinite(s), s, np.nan)          # keep NaNs from diverging runs off the plot
    plt.plot(smooth(np.nan_to_num(s, nan=s[np.isfinite(s)].max() if np.isfinite(s).any() else 3)),
             color=c, label=f'lr={lr}')
plt.xlabel('minibatch step'); plt.ylabel('training loss (smoothed)')
plt.title('Station 4 — learning-rate sweep'); plt.legend(); plt.grid(alpha=0.3); plt.ylim(0, 3)
plt.show()
print('lr=1e-3 crawls; lr≈0.05–0.5 descend well; lr=2.0 is unstable — the classic Goldilocks curve.')

In [ ]:
# --- batch noise shrinks like 1/sqrt(b): distance from the full-batch gradient ---
idxg = idx6[:2000]
Xg = torch.stack([train_full[i][0] for i in idxg]).view(2000, -1).to(device)
Yg = torch.tensor([int(train_full[i][1]) for i in idxg]).to(device)

torch.manual_seed(0)
ref_model = DeepMLP([784, 128, 10], init='he').to(device)
def flat_grad(idx):
    ref_model.zero_grad()
    F.cross_entropy(ref_model(Xg[idx]), Yg[idx]).backward()
    return torch.cat([p.grad.flatten() for p in ref_model.parameters()]).detach()

g_full = flat_grad(torch.arange(2000, device=device))
batch_sizes = [8, 16, 32, 64, 128, 256, 512]
dists = []
for b in batch_sizes:
    d = []
    for _ in range(30):
        sel = torch.randint(0, 2000, (b,), device=device)
        d.append((flat_grad(sel) - g_full).norm().item())
    dists.append(np.mean(d))

plt.figure(figsize=(7.5,5))
plt.loglog(batch_sizes, dists, 'o-', color='crimson', label='measured ‖ĝ_b − g_full‖')
plt.loglog(batch_sizes, dists[0]*np.sqrt(batch_sizes[0])/np.sqrt(batch_sizes),
           '--', color='gray', label='∝ 1/√b reference')
plt.xlabel('batch size b'); plt.ylabel('gradient error'); plt.title('Station 4 — minibatch noise ∝ 1/√b')
plt.legend(); plt.grid(alpha=0.3, which='both'); plt.show()
print('Bigger batches -> less noisy gradient (but more compute per step). Small batches explore more.')

In [ ]:
# --- decaying the learning rate tightens the final loss ---
def train_decay(decay, epochs=12, lr0=0.2):
    torch.manual_seed(0)
    m = DeepMLP([784, 256, 10], init='he').to(device)
    opt = torch.optim.SGD(m.parameters(), lr=lr0)
    sched = torch.optim.lr_scheduler.LinearLR(opt, 1.0, 0.02, total_iters=epochs) if decay else None
    losses = []
    for ep in range(epochs):
        m.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); F.cross_entropy(m(xb), yb).backward(); opt.step()
        if sched: sched.step()
        losses.append(evaluate(m, train_loader)[0])
    return losses

plt.figure(figsize=(7.5,4.6))
plt.plot(train_decay(False), 'o-', label='constant lr=0.2')
plt.plot(train_decay(True),  'o-', label='decayed 0.2 → ~0')
plt.xlabel('epoch'); plt.ylabel('training loss'); plt.title('Station 4 — a decay schedule settles the run')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print('A fixed rate leaves the iterate rattling in a cloud; shrinking ε tightens it toward a point.')

## Station 5 — Momentum & Nesterov

Plain SGD wastes steps bouncing between the steep walls of a stretched valley. **Momentum** gives the iterate *velocity* — a running average of past gradients — so consistent directions build speed while oscillating ones cancel:

$$v\leftarrow\beta v-\varepsilon\nabla J(\theta),\qquad \theta\leftarrow\theta+v.$$

At `β≈0.9` the effective step along a consistent direction is up to `1/(1−β)≈10×` the raw one. **Nesterov** evaluates the gradient at the look-ahead point `θ+βv`, braking earlier. First the 2-D race, then MNIST.


In [ ]:
def race_2d(kappa=20.0, lr=0.02, beta=0.9, steps=60, start=(-4.0, 4.0)):
    H = np.array([kappa, 1.0])
    def run(kind):
        w = np.array(start, float); v = np.zeros(2); path=[w.copy()]
        for _ in range(steps):
            if kind == 'sgd':
                w = w - lr*(H*w)
            elif kind == 'momentum':
                v = beta*v - lr*(H*w); w = w + v
            else:  # nesterov: gradient at look-ahead point
                g = H*(w + beta*v); v = beta*v - lr*g; w = w + v
            path.append(w.copy())
        return np.array(path)
    return run('sgd'), run('momentum'), run('nesterov')

sgd_p, mom_p, nes_p = race_2d()
g = np.linspace(-5,5,200); W1,W2 = np.meshgrid(g,g); J = 0.5*(20*W1**2 + W2**2)
plt.figure(figsize=(7,6))
plt.contour(W1,W2,J, levels=np.logspace(-1,3,18), cmap='Greys', alpha=0.5)
for p, c, lab in [(sgd_p,'gray','SGD'),(mom_p,'purple','momentum'),(nes_p,'teal','Nesterov')]:
    plt.plot(p[:,0], p[:,1], 'o-', ms=2, lw=1, color=c, label=lab)
plt.scatter(0,0,color='k',zorder=5); plt.legend()
plt.title('Station 5 — same lr & β: momentum/Nesterov glide, SGD zig-zags')
plt.xlabel('w₁ (steep)'); plt.ylabel('w₂ (flat)'); plt.show()

In [ ]:
# --- momentum on MNIST: faster, smoother descent ---
plt.figure(figsize=(8,5))
for mom, nes, c, lab in [(0.0, False,'gray','SGD (β=0)'),
                         (0.9, False,'purple','momentum β=0.9'),
                         (0.9, True, 'teal','Nesterov β=0.9')]:
    torch.manual_seed(0)
    m = DeepMLP([784, 256, 10], init='he')
    h = train(m, train_loader, epochs=10, opt_name='sgd', lr=0.05,
              momentum=mom, nesterov=nes, val_loader=val_loader)
    plt.plot(h['val_acc'], 'o-', color=c, label=f'{lab} (val {h["val_acc"][-1]:.3f})')
plt.xlabel('epoch'); plt.ylabel('validation accuracy')
plt.title('Station 5 — momentum reaches good accuracy in fewer epochs')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print('"SGD" in papers almost always means SGD+momentum, β≈0.9; Nesterov is a free drop-in upgrade.')

## Station 6 — Parameter initialization

A deep net is exquisitely sensitive to its start. Initialization has two jobs:

- **Break symmetry.** If two units in a layer start identical and see the same input, they get identical gradients and stay clones forever. *Never initialize all weights to zero.*
- **Preserve variance.** Scale the random weights so a signal's variance stays roughly constant layer to layer — the antidote to station 3. **Xavier/Glorot** (`Var=2/(n_in+n_out)`) suits tanh; **He** (`Var=2/n_in`) suits ReLU, its factor 2 compensating for ReLU zeroing half its inputs.


In [ ]:
# --- symmetry breaking: zero-init cannot learn ---
res = {}
for init, lab in [('zero', 'all-zero init'), ('he', 'He random init')]:
    torch.manual_seed(0)
    m = DeepMLP([784, 128, 128, 10], act='relu', init=init)
    h = train(m, train_loader, epochs=6, opt_name='sgd', lr=0.1, momentum=0.9, val_loader=val_loader)
    res[lab] = h['val_acc'][-1]
    # measure how identical the hidden units are (std across rows of layer-1 weights)
    row_std = m.lins[0].weight.detach().std(dim=0).mean().item()
    print(f'{lab:16s}: final val acc {h["val_acc"][-1]:.3f} | between-unit weight spread {row_std:.4f}')
print('\nZero init: units stay identical (spread ~0) -> stuck near chance (~0.1). Random init learns.')

In [ ]:
# --- variance propagation: activation std across 12 layers vs init scale ---
sizes = [784] + [128]*11 + [10]
xb, _ = next(iter(train_loader)); xb = xb.to(device)
plt.figure(figsize=(8.5,5))
for scale, c in [(0.5,'navy'), (1.0,'green'), (1.8,'crimson')]:
    torch.manual_seed(0)
    m = DeepMLP(sizes, act='relu', init='he').to(device); m.reset('he', scale=scale)
    with torch.no_grad():
        _, acts = m(xb, return_acts=True)
    stds = [a.std().item() for a in acts[:-1]]
    plt.semilogy(range(1, len(stds)+1), stds, 'o-', color=c, label=f'init scale ×{scale}')
plt.xlabel('layer'); plt.ylabel('activation std (log)'); plt.title('Station 6 — variance propagation')
plt.legend(); plt.grid(alpha=0.3, which='both'); plt.show()
print('×1.0 (He) stays roughly flat; ×0.5 collapses toward 0 (vanishing); ×1.8 blows up (exploding).')

## Station 7 — Adaptive learning rates: AdaGrad, RMSProp, Adam

A single global rate is a compromise — too big for steep parameters, too small for flat ones. Adaptive methods give **each parameter its own step**, scaling it down where gradients are large/frequent and up where they're rare:

$$\text{AdaGrad: } r\!\mathrel{+}=\!g^2\quad\text{RMSProp: } r\leftarrow\rho r+(1-\rho)g^2\quad\text{step}=-\varepsilon\,g/(\sqrt r+\delta)$$
$$\text{Adam: } s\leftarrow\beta_1 s+(1-\beta_1)g,\ r\leftarrow\beta_2 r+(1-\beta_2)g^2,\ \theta\leftarrow\theta-\varepsilon\,\hat s/(\sqrt{\hat r}+\delta).$$

AdaGrad's denominator only grows, so it can stall; RMSProp's *decaying* average fixes that; Adam adds momentum + bias-correction and became the default. First the 2-D race (hand-coded optimizers), then MNIST.


In [ ]:
def opt_race(lr=0.16, steps=60, start=(-4.0, -4.0), kappa=12.0):
    H = np.array([1.0, kappa])                      # flat w1, steep w2
    d = 1e-8
    def run(kind):
        w = np.array(start, float); path=[w.copy()]
        r = np.zeros(2); s = np.zeros(2)
        for t in range(1, steps+1):
            g = H*w
            if kind == 'sgd':
                w = w - lr*g
            elif kind == 'adagrad':
                r += g*g; w = w - lr*g/(np.sqrt(r)+d)
            elif kind == 'rmsprop':
                r = 0.9*r + 0.1*g*g; w = w - lr*g/(np.sqrt(r)+d)
            else:  # adam
                s = 0.9*s + 0.1*g; r = 0.999*r + 0.001*g*g
                sh = s/(1-0.9**t); rh = r/(1-0.999**t)
                w = w - lr*sh/(np.sqrt(rh)+d)
            path.append(w.copy())
        return np.array(path)
    return {k: run(k) for k in ['sgd','adagrad','rmsprop','adam']}

def steps_to_min(path, radius=0.2):
    d = np.linalg.norm(path, axis=1); hit = np.where(d < radius)[0]
    return int(hit[0]) if len(hit) else None

paths = opt_race()
g = np.linspace(-5,5,200); W1,W2 = np.meshgrid(g,g); J = 0.5*(W1**2 + 12*W2**2)
plt.figure(figsize=(7,6))
plt.contour(W1,W2,J, levels=np.logspace(-1,3,18), cmap='Greys', alpha=0.5)
for k, c in [('sgd','gray'),('adagrad','orange'),('rmsprop','teal'),('adam','crimson')]:
    n = steps_to_min(paths[k]); tag = f'{n} steps' if n else 'did not reach'
    plt.plot(paths[k][:,0], paths[k][:,1], 'o-', ms=2, lw=1, color=c, label=f'{k} ({tag})')
plt.scatter(0,0,color='k',zorder=5,label='minimum'); plt.legend()
plt.xlim(-5,5); plt.ylim(-5,5)
plt.title('Station 7 — adaptive methods reach the star; SGD zig-zags the steep axis')
plt.xlabel('w₁ (flat)'); plt.ylabel('w₂ (steep)'); plt.show()
print('Same shared lr: SGD ping-pongs in the steep w₂ direction and reaches the min later;')
print('RMSProp/Adam rescale per-axis and drive in straighter. AdaGrad stalls as its rate decays.')

In [ ]:
# --- AdaGrad's effective rate decays toward zero (its fatal flaw) ---
g_stream = np.random.RandomState(0).randn(300)*1.0 + 0.5     # a noisy but persistent gradient
r_ada, r_rms = 0.0, 0.0; eff_ada, eff_rms = [], []
for g in g_stream:
    r_ada += g*g;                         eff_ada.append(0.1/(np.sqrt(r_ada)+1e-8))
    r_rms = 0.9*r_rms + 0.1*g*g;          eff_rms.append(0.1/(np.sqrt(r_rms)+1e-8))
plt.figure(figsize=(7.5,4.6))
plt.plot(eff_ada, color='orange', label='AdaGrad effective rate → 0')
plt.plot(eff_rms, color='teal',   label='RMSProp effective rate stabilizes')
plt.xlabel('step'); plt.ylabel('effective per-parameter rate'); plt.yscale('log')
plt.title('Station 7 — why RMSProp replaced AdaGrad'); plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# --- optimizer bake-off on MNIST ---
setups = [('sgd',0.05,0.0,'gray','SGD'),
          ('sgd',0.05,0.9,'purple','SGD+momentum'),
          ('rmsprop',1e-3,0.0,'teal','RMSProp'),
          ('adam',1e-3,0.0,'crimson','Adam'),
          ('adamw',1e-3,0.0,'navy','AdamW')]
plt.figure(figsize=(8.5,5))
for name, lr, mom, c, lab in setups:
    torch.manual_seed(0)
    m = DeepMLP([784, 256, 128, 10], init='he')
    h = train(m, train_loader, epochs=10, opt_name=name, lr=lr, momentum=mom,
              weight_decay=(1e-2 if name=='adamw' else 0.0), val_loader=val_loader)
    plt.plot(h['val_acc'], 'o-', color=c, label=f'{lab} (val {h["val_acc"][-1]:.3f})')
plt.xlabel('epoch'); plt.ylabel('validation accuracy'); plt.title('Station 7 — optimizer bake-off on MNIST')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()
print('Adam/AdamW converge fast with little tuning — the "3e-4 and Adam" default. Tuned SGD+momentum can still win on vision.')

## Station 8 — Greedy layer-wise pretraining

Before good init and activations tamed deep training, the trick was: don't train the deep net all at once. Train it **one layer at a time** — each layer (as an autoencoder) learning to represent the layer below — then stack and **fine-tune** end-to-end. Two benefits: **better optimization** (each layer starts in a sensible region) and **regularization** (unsupervised structure transfers to the task), *most valuable when labels are scarce*.

We pretrain two autoencoder layers on unlabeled images, use their encoders to initialize a classifier, and fine-tune on just **500 labels** — versus the same classifier from random init.


In [ ]:
class AE(nn.Module):
    def __init__(self, d_in, d_hid):
        super().__init__(); self.enc = nn.Linear(d_in, d_hid); self.dec = nn.Linear(d_hid, d_in)
    def forward(self, x):
        h = torch.relu(self.enc(x)); return self.dec(h), h

def train_ae(ae, feats, epochs=10, lr=1e-3):
    ae.to(device); opt = torch.optim.Adam(ae.parameters(), lr=lr)
    dl = DataLoader(TensorDataset(feats), batch_size=128, shuffle=True)
    for ep in range(epochs):
        for (xb,) in dl:
            opt.zero_grad(); recon, _ = ae(xb); F.mse_loss(recon, xb).backward(); opt.step()
    return ae

# unlabeled features = the 6000-image subset, flattened
unlab = torch.stack([train_full[i][0] for i in idx6]).view(len(idx6), -1).to(device)
torch.manual_seed(0)
ae1 = train_ae(AE(784, 256), unlab)                                   # layer 1
with torch.no_grad(): feat1 = torch.relu(ae1.enc(unlab))
ae2 = train_ae(AE(256, 64), feat1)                                    # layer 2 on layer-1 codes
print('Pretrained two autoencoder layers (784→256→64) on unlabeled images.')

In [ ]:
def finetune(pretrained, n_labels=500, epochs=25, lr=0.05):
    torch.manual_seed(1)
    clf = DeepMLP([784, 256, 64, 10], act='relu', init='he')
    if pretrained:                                    # copy encoder weights from the AEs
        clf.lins[0].weight.data.copy_(ae1.enc.weight.data); clf.lins[0].bias.data.copy_(ae1.enc.bias.data)
        clf.lins[1].weight.data.copy_(ae2.enc.weight.data); clf.lins[1].bias.data.copy_(ae2.enc.bias.data)
    lab_loader = make_loader(n_labels, bs=64, seed=7)
    h = train(clf, lab_loader, epochs=epochs, opt_name='sgd', lr=lr, momentum=0.9, val_loader=val_loader)
    return h['val_acc'][-1]

acc_rand = finetune(pretrained=False)
acc_pre  = finetune(pretrained=True)
print(f'500 labels · random init      : val acc {acc_rand:.3f}')
print(f'500 labels · AE-pretrained init: val acc {acc_pre:.3f}')
print(f'\nPretraining edge in the low-label regime: {acc_pre - acc_rand:+.3f}')

In [ ]:
# The pretraining edge shrinks as labels become plentiful.
counts = [200, 500, 2000]
rand_accs = [finetune(False, n) for n in counts]
pre_accs  = [finetune(True,  n) for n in counts]
plt.figure(figsize=(7.5,5))
plt.plot(counts, rand_accs, 'o-', color='gray',    label='random init')
plt.plot(counts, pre_accs,  'o-', color='crimson', label='AE-pretrained init')
plt.xscale('log'); plt.xlabel('number of labeled examples'); plt.ylabel('validation accuracy')
plt.title('Station 8 — pretraining helps most when labels are scarce')
plt.legend(); plt.grid(alpha=0.3, which='both'); plt.show()
print("Its idea never left: today it's transfer learning & self-supervised pretraining, scaled up.")

## Station 9 — Adversarial training

Even an accurate network can be fooled by a perturbation too small to see. **Adversarial examples** exploit the network's near-linear behaviour in high dimensions: many tiny, aligned nudges add up. The one-step **Fast Gradient Sign Method** pushes every pixel in the direction that most increases the loss:

$$x_{adv}=x+\varepsilon\cdot\mathrm{sign}\big(\nabla_x L(f(x),y)\big).$$

**Adversarial training** generates these attacks during training and forces the model to classify them correctly — hardening it and, as a bonus, smoothing the decision surface. We train a clean model, break it with FGSM, then train an adversarial model and compare.


In [ ]:
def fgsm(model, x, y, eps):
    x = x.clone().detach().to(device).requires_grad_(True)
    F.cross_entropy(model(x), y.to(device)).backward()
    return (x + eps * x.grad.sign()).detach()

@torch.no_grad()
def clean_acc(model, loader):
    return evaluate(model, loader)[1]

def adv_acc(model, loader, eps):
    model.eval(); tot = correct = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        xadv = fgsm(model, xb, yb, eps)
        with torch.no_grad():
            correct += (model(xadv).argmax(1) == yb).sum().item(); tot += yb.size(0)
    return correct / tot

# clean-trained model
torch.manual_seed(0)
clean_model = DeepMLP([784, 256, 128, 10], init='he')
train(clean_model, train_loader, epochs=8, opt_name='adam', lr=1e-3, val_loader=None)
small_val = make_loader(2000, bs=256, seed=3, train=False)
print(f'clean model — clean acc {clean_acc(clean_model, small_val):.3f}')
for eps in [0.05, 0.1, 0.2, 0.3]:
    print(f'  FGSM ε={eps}: adversarial acc {adv_acc(clean_model, small_val, eps):.3f}')

In [ ]:
# Visualize one attack: clean image, perturbation, adversarial image.
xb, yb = next(iter(small_val)); x0, y0 = xb[:1].to(device), yb[:1].to(device)
xadv = fgsm(clean_model, x0, y0, eps=0.25)
pred_clean = clean_model(x0).argmax(1).item()
pred_adv   = clean_model(xadv).argmax(1).item()
imgs = [x0, (xadv - x0), xadv]
titles = [f'clean → pred {pred_clean}', 'perturbation (×sign)', f'adversarial → pred {pred_adv}']
plt.figure(figsize=(10,3.4))
for i,(im,t) in enumerate(zip(imgs, titles)):
    plt.subplot(1,3,i+1); plt.imshow(im.detach().cpu().view(28,28), cmap='gray'); plt.title(t, fontsize=10); plt.axis('off')
plt.suptitle(f'Station 9 — a tiny push flips the label (true digit {y0.item()})'); plt.tight_layout(); plt.show()

In [ ]:
# Adversarial training: mix FGSM examples into each batch.
def train_adversarial(eps=0.2, epochs=8, lr=1e-3):
    torch.manual_seed(0)
    m = DeepMLP([784, 256, 128, 10], init='he').to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    for ep in range(epochs):
        m.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            xadv = fgsm(m, xb, yb, eps)                       # attack the current model
            opt.zero_grad()
            loss = 0.5*F.cross_entropy(m(xb), yb) + 0.5*F.cross_entropy(m(xadv), yb)
            loss.backward(); opt.step()
    return m

adv_model = train_adversarial()
eps_grid = [0.0, 0.05, 0.1, 0.2, 0.3]
clean_curve = [clean_acc(clean_model, small_val) if e==0 else adv_acc(clean_model, small_val, e) for e in eps_grid]
adv_curve   = [clean_acc(adv_model,   small_val) if e==0 else adv_acc(adv_model,   small_val, e) for e in eps_grid]
plt.figure(figsize=(7.5,5))
plt.plot(eps_grid, clean_curve, 'o-', color='gray',    label='clean-trained model')
plt.plot(eps_grid, adv_curve,   'o-', color='crimson', label='adversarially-trained model')
plt.xlabel('FGSM perturbation ε'); plt.ylabel('accuracy under attack')
plt.title('Station 9 — adversarial training buys robustness'); plt.legend(); plt.grid(alpha=0.3); plt.show()
print('The clean model collapses as ε grows; the adversarially-trained one degrades far more gracefully.')

## Key takeaways — carry these off the map

1. **Optimize a proxy.** We minimize empirical risk with a smooth *surrogate* (cross-entropy), then early-stop on a *validation* metric — not the training loss.
2. **Saddles, not minima.** The real obstacles are saddle points, plateaus, ill-conditioning and cliffs; most local minima are nearly as good as the global one.
3. **Depth multiplies signals.** Gradients scale like `(w·σ′)^L`; below 1 they vanish, above 1 they explode. ReLU, good init, normalization and residuals keep the factor near 1.
4. **SGD: cheap, noisy steps.** Minibatch gradients give many fast steps; the noise (∝ 1/√b) aids exploration but forces a decaying learning rate to settle.
5. **Momentum builds velocity.** `v ← βv − ε∇J` accelerates consistent directions and cancels oscillation; Nesterov looks ahead to brake earlier.
6. **Init breaks symmetry & keeps variance.** Random weights (never all-zero), scaled by fan-in — Xavier for tanh, He for ReLU — preserve the signal through depth.
7. **Adam: a rate per parameter.** RMSProp's decaying gradient-square average plus momentum, bias-corrected — the forgiving default; tuned SGD+momentum can still win on vision.
8. **Pretrain, then adapt.** Greedy layer-wise pretraining gave deep nets a good start; the idea lives on as transfer and self-supervised learning.
9. **Adversarial hardening.** Tiny gradient-aligned nudges (FGSM) flip predictions; training on them buys robustness and smooths the decision surface.

### Try it yourself
- In station 4, push the batch-noise experiment onto a *trained* model — does the 1/√b law still hold?
- In station 7, add a learning-rate warmup before the Adam bake-off and see which curves change.
- In station 9, replace one-step FGSM with a few-step PGD attack (iterate the sign step with small steps) and re-measure robustness.

*Reference: I. Goodfellow, Y. Bengio, A. Courville, "Deep Learning" (MIT Press), Chapter 8 (with the adversarial section from §7.13).*
